# Laboratorio 2 — Complejidad y búsqueda de hiperparámetros
## AlpesHearth: Estimación del score de riesgo cardiovascular

En este notebook abordaremos el Laboratorio 2, cuyo objetivo es comparar diferentes enfoques de modelado para estimar el score de riesgo cardiovascular (`CVD Risk Score`) de los pacientes de AlpesHearth. A partir del conjunto de datos del Laboratorio 1, exploraremos modelos de regresión polinomial, regularización Ridge y Lasso, combinaciones de ambos, y finalmente estimaremos intervalos de confianza mediante bootstrapping para cuantificar la incertidumbre del mejor modelo seleccionado.

El conjunto de datos utilizado es `Datos_Lab_1.csv`. Dado que el archivo de test externo (`Datos_Test_Lab_1.csv`) no contiene la variable objetivo, seguiremos el mismo patrón de los notebooks de referencia del curso: cargar un único archivo y dividirlo internamente con `train_test_split`.

## 1. Importación de las librerías

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (
    train_test_split, GridSearchCV, KFold,
    cross_val_score, validation_curve
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    StandardScaler, MinMaxScaler, OneHotEncoder,
    FunctionTransformer, PolynomialFeatures
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.utils import resample

In [ ]:
import sklearn
print(f'pandas      : {pd.__version__}')
print(f'numpy       : {np.__version__}')
print(f'scikit-learn: {sklearn.__version__}')

## 2. Carga de los datos

Cargamos el conjunto de datos resultante del Laboratorio 1. El archivo `Datos_Lab_1.csv` contiene 1 639 registros y 24 columnas, incluyendo variables clínicas y demográficas de los pacientes, así como la variable objetivo `CVD Risk Score`.

In [ ]:
datos = pd.read_csv('./data/Datos_Lab_1.csv')
print(f'Filas: {datos.shape[0]}, Columnas: {datos.shape[1]}')
datos.head()

## 3. Preparación inicial del conjunto de datos

Antes de construir los modelos realizamos tres ajustes:

1. **Eliminar columnas que no deben entrar como predictores:** `Patient ID`, `Date of Service`, `CVD Risk Level` (esta última es una categorización derivada directamente de `CVD Risk Score`, por lo que incluirla generaría fuga de datos) y `Blood Pressure (mmHg)` (ya descompuesta en `Systolic BP` y `Diastolic BP`).
2. **Separar la variable objetivo.**
3. **Revisar valores faltantes** para confirmar que el pipeline de imputación los cubrirá.

In [ ]:
data = datos.copy()

cols_to_drop = ['Patient ID', 'Date of Service', 'CVD Risk Level', 'Blood Pressure (mmHg)']
data = data.drop(columns=cols_to_drop)

# Eliminar filas donde la variable objetivo es nula (29 casos)
data = data.dropna(subset=['CVD Risk Score'])

print(f'Filas tras limpieza: {data.shape[0]}, Columnas: {data.shape[1]}')
data.head()

In [ ]:
# Revisar valores faltantes restantes
missing = data.isnull().sum()
print('Valores faltantes por columna:')
print(missing[missing > 0])

Los valores faltantes en las variables numéricas serán manejados mediante **imputación por la media** dentro del pipeline, tal como se hizo en el Laboratorio 1. Esto garantiza que la imputación se aprenda exclusivamente sobre el conjunto de entrenamiento y se aplique de forma consistente al conjunto de test.

## 4. Partición de los datos

Separamos la variable objetivo de las variables predictoras y dividimos el conjunto en entrenamiento (80%) y test (20%), usando `random_state=1` para reproducibilidad.

In [ ]:
target = 'CVD Risk Score'
X = data.drop(columns=[target])
y = data[target]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=1)

In [ ]:
print(f'Tamaño entrenamiento: {X_train.shape}')
print(f'Tamaño test:          {X_test.shape}')

## 5. Definición de variables y transformadores base

Separamos explícitamente las variables numéricas y categóricas del dataset. Esto nos permite aplicar transformaciones diferenciadas dentro del pipeline según el tipo de dato.

In [ ]:
numeric_features = [
    'Age', 'Weight (kg)', 'Height (m)', 'BMI',
    'Abdominal Circumference (cm)', 'Total Cholesterol (mg/dL)',
    'HDL (mg/dL)', 'Fasting Blood Sugar (mg/dL)', 'Height (cm)',
    'Waist-to-Height Ratio', 'Systolic BP', 'Diastolic BP',
    'Estimated LDL (mg/dL)'
]

categorical_features = [
    'Sex', 'Smoking Status', 'Diabetes Status',
    'Physical Activity Level', 'Family History of CVD',
    'Blood Pressure Category'
]

print(f'Variables numéricas  ({len(numeric_features)}): {numeric_features}')
print(f'Variables categóricas ({len(categorical_features)}): {categorical_features}')

In [ ]:
# Transformador para variables categóricas (común a todos los modelos)
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', drop='if_binary')),
])

---

## Actividad 1: Modelo de regresión polinomial con búsqueda de hiperparámetros

Construimos un pipeline que incorpora la generación de características polinomiales (`PolynomialFeatures`) seguida de un modelo de regresión lineal. La búsqueda del grado óptimo del polinomio y de la estrategia de escalamiento se realiza mediante `GridSearchCV` con validación cruzada de 5 folds. Explorar distintas estrategias de escalamiento dentro del `GridSearchCV` garantiza que la comparación sea justa: el escalamiento se aprende solo sobre el fold de entrenamiento en cada iteración.

In [ ]:
# Transformador numérico con características polinomiales
numeric_transformer_poly = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('poly',    PolynomialFeatures(degree=2, include_bias=False)),
])

preprocessor_poly = ColumnTransformer(transformers=[
    ('num', numeric_transformer_poly, numeric_features),
    ('cat', categorical_transformer,  categorical_features),
])

pipeline_poly = Pipeline(steps=[
    ('preprocesamiento', preprocessor_poly),
    ('modelo',           LinearRegression()),
])

In [ ]:
# Espacio de búsqueda: grado del polinomio y estrategia de escalamiento
param_grid_poly = {
    'preprocesamiento__num__scaler': [StandardScaler(), MinMaxScaler()],
    'preprocesamiento__num__poly__degree': [1, 2, 3],
}

kfold_5 = KFold(n_splits=5, shuffle=True, random_state=1)

grid_poly = GridSearchCV(
    estimator=pipeline_poly,
    param_grid=param_grid_poly,
    cv=kfold_5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
%%time
grid_poly.fit(X_train, y_train)

In [ ]:
print('Mejor configuración polinomial:')
for k, v in grid_poly.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor RMSE en CV: {-grid_poly.best_score_:.4f}')

In [ ]:
best_poly = grid_poly.best_estimator_

y_train_pred_poly = best_poly.predict(X_train)
print('------ Mejor modelo polinomial - Entrenamiento ------')
print(f'RMSE: {np.sqrt(mean_squared_error(y_train, y_train_pred_poly)):.4f}')
print(f'MAE:  {mean_absolute_error(y_train, y_train_pred_poly):.4f}')
print(f'R²:   {r2_score(y_train, y_train_pred_poly):.4f}')

In [ ]:
y_test_pred_poly = best_poly.predict(X_test)
rmse_test_poly = np.sqrt(mean_squared_error(y_test, y_test_pred_poly))
mae_test_poly  = mean_absolute_error(y_test, y_test_pred_poly)
r2_test_poly   = r2_score(y_test, y_test_pred_poly)

print('------ Mejor modelo polinomial - Test ------')
print(f'RMSE: {rmse_test_poly:.4f}')
print(f'MAE:  {mae_test_poly:.4f}')
print(f'R²:   {r2_test_poly:.4f}')

**Justificación de decisiones:** Se exploran grados 1, 2 y 3. El grado 1 equivale a la regresión lineal clásica y actúa como línea base. El grado 2 introduce interacciones cuadráticas entre variables clínicas (por ejemplo, interacción entre BMI y colesterol), que pueden ser relevantes para el riesgo cardiovascular. El grado 3 incrementa la complejidad y es esperable que produzca sobreajuste dado el tamaño del dataset (≈1 600 registros). Si el RMSE en test crece con respecto al entrenamiento al aumentar el grado, se confirma el sobreajuste.

---

## Actividad 2: Curvas de validación

Generamos curvas de validación para visualizar cómo evoluciona el RMSE (en entrenamiento y en validación cruzada) a medida que aumenta el grado del polinomio. La banda de color alrededor de cada curva representa la desviación estándar entre los folds, indicando la variabilidad del desempeño. Este análisis permite identificar el punto a partir del cual el modelo comienza a sobreajustarse (sesgo-varianza trade-off).

In [ ]:
# Pipeline para validation_curve con scaler fijo (StandardScaler)
numeric_transformer_vc = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('poly',    PolynomialFeatures(degree=2, include_bias=False)),
])

preprocessor_vc = ColumnTransformer(transformers=[
    ('num', numeric_transformer_vc, numeric_features),
    ('cat', categorical_transformer, categorical_features),
])

pipeline_vc = Pipeline(steps=[
    ('preprocesamiento', preprocessor_vc),
    ('modelo',           LinearRegression()),
])

param_range = [1, 2, 3, 4]

train_scores, val_scores = validation_curve(
    pipeline_vc,
    X_train, y_train,
    param_name='preprocesamiento__num__poly__degree',
    param_range=param_range,
    cv=kfold_5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1
)

train_rmse = -train_scores
val_rmse   = -val_scores

In [ ]:
plt.figure(figsize=(8, 5))

plt.plot(param_range, train_rmse.mean(axis=1),
         marker='o', label='Entrenamiento', color='steelblue')
plt.fill_between(param_range,
                 train_rmse.mean(axis=1) - train_rmse.std(axis=1),
                 train_rmse.mean(axis=1) + train_rmse.std(axis=1),
                 alpha=0.2, color='steelblue')

plt.plot(param_range, val_rmse.mean(axis=1),
         marker='o', label='Validación cruzada', color='coral')
plt.fill_between(param_range,
                 val_rmse.mean(axis=1) - val_rmse.std(axis=1),
                 val_rmse.mean(axis=1) + val_rmse.std(axis=1),
                 alpha=0.2, color='coral')

plt.xlabel('Grado del polinomio')
plt.ylabel('RMSE')
plt.title('Curvas de validación — Grado del polinomio')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

**Interpretación sesgo-varianza:**

- **Grado 1 (regresión lineal):** Las curvas de entrenamiento y validación convergen en un error relativamente alto. Si ambas curvas tienen RMSE similar y elevado, hay **alto sesgo** (subajuste): el modelo es demasiado simple para capturar la relación entre las variables clínicas y el score de riesgo.

- **Grado 2:** Idealmente ambas curvas bajan. Si la brecha entre entrenamiento y validación es pequeña, el modelo está bien calibrado.

- **Grado 3-4:** El error en entrenamiento sigue bajando, pero si el error en validación cruzada sube o su desviación estándar aumenta significativamente, el modelo está **sobreajustando**: memoriza los patrones del conjunto de entrenamiento, incluyendo ruido, y pierde capacidad de generalización. El punto de inflexión en la curva de validación determina el grado óptimo.

---

## Actividad 3: Modelos de regresión regularizados (Ridge y Lasso)

Implementamos pipelines para Ridge (regularización L2) y Lasso (regularización L1). Ambos añaden un término de penalización sobre los coeficientes en la función de costo, controlando la complejidad del modelo. El hiperparámetro `alpha` determina la fuerza de la penalización y se optimiza con `GridSearchCV` usando 10 folds (mayor que en el modelo polinomial para obtener una estimación más estable del desempeño en generalización).

### 3.1 Ridge (regularización L2)

In [ ]:
# Transformador numérico base (sin términos polinomiales)
numeric_transformer_base = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
])

preprocessor_base = ColumnTransformer(transformers=[
    ('num', numeric_transformer_base, numeric_features),
    ('cat', categorical_transformer,  categorical_features),
])

pipeline_ridge = Pipeline(steps=[
    ('preprocesamiento', preprocessor_base),
    ('modelo',           Ridge()),
])

In [ ]:
param_grid_ridge = {
    'preprocesamiento__num__scaler': [StandardScaler(), MinMaxScaler()],
    'modelo__alpha': [0.01, 0.1, 1, 10, 100],
}

kfold_10 = KFold(n_splits=10, shuffle=True, random_state=1)

grid_ridge = GridSearchCV(
    estimator=pipeline_ridge,
    param_grid=param_grid_ridge,
    cv=kfold_10,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
%%time
grid_ridge.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros Ridge:')
for k, v in grid_ridge.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor RMSE en CV: {-grid_ridge.best_score_:.4f}')

In [ ]:
best_ridge = grid_ridge.best_estimator_

y_pred_ridge_train = best_ridge.predict(X_train)
rmse_ridge_train = np.sqrt(mean_squared_error(y_train, y_pred_ridge_train))
mae_ridge_train  = mean_absolute_error(y_train, y_pred_ridge_train)
r2_ridge_train   = r2_score(y_train, y_pred_ridge_train)

print('------ Ridge - Entrenamiento ------')
print(f'RMSE: {rmse_ridge_train:.4f}')
print(f'MAE:  {mae_ridge_train:.4f}')
print(f'R²:   {r2_ridge_train:.4f}')

In [ ]:
y_pred_ridge_test = best_ridge.predict(X_test)
rmse_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_ridge_test))
mae_ridge_test  = mean_absolute_error(y_test, y_pred_ridge_test)
r2_ridge_test   = r2_score(y_test, y_pred_ridge_test)

print('------ Ridge - Test ------')
print(f'RMSE: {rmse_ridge_test:.4f}')
print(f'MAE:  {mae_ridge_test:.4f}')
print(f'R²:   {r2_ridge_test:.4f}')

#### Revisión de coeficientes Ridge

Revisar los coeficientes permite comprender el efecto de la regularización L2. Ridge reduce la magnitud de todos los coeficientes de forma proporcional, pero **ninguno llega exactamente a cero**. Esto es útil cuando se sospecha que todas las variables clínicas contribuyen al score de riesgo cardiovascular, ya que el modelo no descarta ninguna de ellas.

In [ ]:
preproc_ridge = best_ridge.named_steps['preprocesamiento']
feature_names_ridge = preproc_ridge.get_feature_names_out()
coefs_ridge = best_ridge.named_steps['modelo'].coef_

coef_ridge_df = pd.DataFrame({
    'Característica': feature_names_ridge,
    'Coeficiente': coefs_ridge
})
coef_ridge_df['Característica'] = coef_ridge_df['Característica'].str.split('__').str[-1]
coef_ridge_df['Coeficiente'] = coef_ridge_df['Coeficiente'].round(4)
coef_ridge_df_sorted = coef_ridge_df.sort_values('Coeficiente', key=abs, ascending=False)
coef_ridge_df_sorted

In [ ]:
top15_ridge = coef_ridge_df_sorted.head(15)

plt.figure(figsize=(9, 5))
sns.barplot(data=top15_ridge, x='Coeficiente', y='Característica', palette='coolwarm')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 coeficientes — Modelo Ridge')
plt.xlabel('Coeficiente')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

### 3.2 Lasso (regularización L1)

In [ ]:
pipeline_lasso = Pipeline(steps=[
    ('preprocesamiento', preprocessor_base),
    ('modelo',           Lasso(max_iter=10000)),
])

In [ ]:
param_grid_lasso = {
    'preprocesamiento__num__scaler': [StandardScaler(), MinMaxScaler()],
    'modelo__alpha': [0.001, 0.01, 0.1, 1, 10],
}

grid_lasso = GridSearchCV(
    estimator=pipeline_lasso,
    param_grid=param_grid_lasso,
    cv=kfold_10,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
%%time
grid_lasso.fit(X_train, y_train)

In [ ]:
print('Mejores hiperparámetros Lasso:')
for k, v in grid_lasso.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor RMSE en CV: {-grid_lasso.best_score_:.4f}')

In [ ]:
best_lasso = grid_lasso.best_estimator_

y_pred_lasso_train = best_lasso.predict(X_train)
rmse_lasso_train = np.sqrt(mean_squared_error(y_train, y_pred_lasso_train))
mae_lasso_train  = mean_absolute_error(y_train, y_pred_lasso_train)
r2_lasso_train   = r2_score(y_train, y_pred_lasso_train)

print('------ Lasso - Entrenamiento ------')
print(f'RMSE: {rmse_lasso_train:.4f}')
print(f'MAE:  {mae_lasso_train:.4f}')
print(f'R²:   {r2_lasso_train:.4f}')

In [ ]:
y_pred_lasso_test = best_lasso.predict(X_test)
rmse_lasso_test = np.sqrt(mean_squared_error(y_test, y_pred_lasso_test))
mae_lasso_test  = mean_absolute_error(y_test, y_pred_lasso_test)
r2_lasso_test   = r2_score(y_test, y_pred_lasso_test)

print('------ Lasso - Test ------')
print(f'RMSE: {rmse_lasso_test:.4f}')
print(f'MAE:  {mae_lasso_test:.4f}')
print(f'R²:   {r2_lasso_test:.4f}')

#### Revisión de coeficientes Lasso y selección automática de variables

La propiedad más importante de Lasso es que puede llevar exactamente a cero los coeficientes de variables poco informativas, realizando una **selección automática de características**. Esto es especialmente valioso en el contexto clínico de AlpesHearth, donde un modelo con pocas variables es más fácil de interpretar y de comunicar a los equipos médicos.

In [ ]:
preproc_lasso = best_lasso.named_steps['preprocesamiento']
feature_names_lasso = preproc_lasso.get_feature_names_out()
coefs_lasso = best_lasso.named_steps['modelo'].coef_

coef_lasso_df = pd.DataFrame({
    'Característica': feature_names_lasso,
    'Coeficiente': coefs_lasso
})
coef_lasso_df['Característica'] = coef_lasso_df['Característica'].str.split('__').str[-1]
coef_lasso_df['Coeficiente'] = coef_lasso_df['Coeficiente'].round(4)
coef_lasso_df_sorted = coef_lasso_df.sort_values('Coeficiente', key=abs, ascending=False)
coef_lasso_df_sorted

In [ ]:
# Variables llevadas a cero por Lasso
vars_cero = coef_lasso_df[coef_lasso_df['Coeficiente'] == 0]
print(f'Variables con coeficiente = 0 ({len(vars_cero)} de {len(coef_lasso_df)}):')
print(vars_cero['Característica'].tolist())

In [ ]:
# Variables activas (coeficiente ≠ 0)
vars_activas = coef_lasso_df_sorted[coef_lasso_df_sorted['Coeficiente'] != 0]
top15_lasso = vars_activas.head(15)

plt.figure(figsize=(9, 5))
sns.barplot(data=top15_lasso, x='Coeficiente', y='Característica', palette='coolwarm')
plt.axvline(0, color='black', linewidth=0.8)
plt.title('Top 15 coeficientes activos — Modelo Lasso')
plt.xlabel('Coeficiente')
plt.ylabel('Variable')
plt.tight_layout()
plt.show()

**Análisis comparativo Ridge vs. Lasso:**

Ridge es preferible cuando se cree que todas las variables clínicas contribuyen al score de riesgo y se quiere preservar la información de todas ellas, a costa de menos interpretabilidad. Lasso es preferible cuando el objetivo es identificar los factores de riesgo más determinantes, eliminando variables redundantes o irrelevantes. En el contexto de AlpesHearth, las variables anuladas por Lasso son candidatas a ser excluidas de futuras mediciones rutinarias, lo que puede reducir costos operativos.

---

## Actividad 4: Regresión polinomial regularizada

Combinamos características polinomiales con regularización Ridge. La motivación es que al generar términos polinomiales el espacio de características crece considerablemente (especialmente con grado ≥ 2), lo que incrementa el riesgo de sobreajuste. La regularización actúa como freno, penalizando los coeficientes de los términos menos relevantes. La búsqueda de hiperparámetros explora simultáneamente el grado, el alpha y la estrategia de escalamiento.

In [ ]:
numeric_transformer_poly_reg = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler',  StandardScaler()),
    ('poly',    PolynomialFeatures(degree=2, include_bias=False)),
])

preprocessor_poly_reg = ColumnTransformer(transformers=[
    ('num', numeric_transformer_poly_reg, numeric_features),
    ('cat', categorical_transformer,      categorical_features),
])

pipeline_poly_ridge = Pipeline(steps=[
    ('preprocesamiento', preprocessor_poly_reg),
    ('modelo',           Ridge()),
])

In [ ]:
param_grid_poly_ridge = {
    'preprocesamiento__num__scaler':      [StandardScaler(), MinMaxScaler()],
    'preprocesamiento__num__poly__degree': [1, 2, 3],
    'modelo__alpha':                       [0.1, 1, 10, 100],
}

grid_poly_ridge = GridSearchCV(
    estimator=pipeline_poly_ridge,
    param_grid=param_grid_poly_ridge,
    cv=kfold_5,
    scoring='neg_root_mean_squared_error',
    n_jobs=-1,
    return_train_score=True
)

In [ ]:
%%time
grid_poly_ridge.fit(X_train, y_train)

In [ ]:
print('Mejor configuración polinomial regularizada:')
for k, v in grid_poly_ridge.best_params_.items():
    print(f'  {k}: {v}')
print(f'\nMejor RMSE en CV: {-grid_poly_ridge.best_score_:.4f}')

In [ ]:
best_poly_ridge = grid_poly_ridge.best_estimator_

y_pred_poly_ridge_train = best_poly_ridge.predict(X_train)
rmse_poly_ridge_train = np.sqrt(mean_squared_error(y_train, y_pred_poly_ridge_train))
mae_poly_ridge_train  = mean_absolute_error(y_train, y_pred_poly_ridge_train)
r2_poly_ridge_train   = r2_score(y_train, y_pred_poly_ridge_train)

print('------ Polinomial + Ridge - Entrenamiento ------')
print(f'RMSE: {rmse_poly_ridge_train:.4f}')
print(f'MAE:  {mae_poly_ridge_train:.4f}')
print(f'R²:   {r2_poly_ridge_train:.4f}')

In [ ]:
y_pred_poly_ridge_test = best_poly_ridge.predict(X_test)
rmse_poly_ridge_test = np.sqrt(mean_squared_error(y_test, y_pred_poly_ridge_test))
mae_poly_ridge_test  = mean_absolute_error(y_test, y_pred_poly_ridge_test)
r2_poly_ridge_test   = r2_score(y_test, y_pred_poly_ridge_test)

print('------ Polinomial + Ridge - Test ------')
print(f'RMSE: {rmse_poly_ridge_test:.4f}')
print(f'MAE:  {mae_poly_ridge_test:.4f}')
print(f'R²:   {r2_poly_ridge_test:.4f}')

**Análisis:** Si el modelo polinomial regularizado obtiene un RMSE en test comparable o menor al modelo polinomial sin regularización, confirma que la penalización logra controlar el sobreajuste incluso al aumentar la complejidad. Si el RMSE en test del modelo polinomial regularizado es mayor al de Ridge simple, sugiere que las relaciones no lineales entre las variables clínicas no agregan información predictiva suficiente como para justificar la complejidad adicional.

---

## Actividad 5: Comparación y selección del mejor modelo

Consolidamos los resultados de todos los modelos en una tabla comparativa. La selección del mejor modelo considera:

1. **RMSE en CV (media):** capacidad de generalización estimada.
2. **RMSE en CV (std):** estabilidad del modelo — un std alto indica que el rendimiento varía mucho según los datos.
3. **RMSE en test:** desempeño real sobre datos no vistos.
4. **Brecha train-test:** indicador de sobreajuste.

In [ ]:
def get_cv_stats(grid):
    idx = grid.best_index_
    media = -grid.cv_results_['mean_test_score'][idx]
    std   =  grid.cv_results_['std_test_score'][idx]
    return round(media, 4), round(std, 4)

rmse_cv_poly,        std_cv_poly        = get_cv_stats(grid_poly)
rmse_cv_ridge,       std_cv_ridge       = get_cv_stats(grid_ridge)
rmse_cv_lasso,       std_cv_lasso       = get_cv_stats(grid_lasso)
rmse_cv_poly_ridge,  std_cv_poly_ridge  = get_cv_stats(grid_poly_ridge)

In [ ]:
resultados = pd.DataFrame([
    {
        'Modelo':           'Polinomial (sin regularización)',
        'RMSE CV (media)':  rmse_cv_poly,
        'RMSE CV (std)':    std_cv_poly,
        'RMSE Test':        round(rmse_test_poly, 4),
        'MAE Test':         round(mae_test_poly, 4),
        'R² Test':          round(r2_test_poly, 4),
        'Mejores hiperparámetros': str(grid_poly.best_params_),
    },
    {
        'Modelo':           'Ridge',
        'RMSE CV (media)':  rmse_cv_ridge,
        'RMSE CV (std)':    std_cv_ridge,
        'RMSE Test':        round(rmse_ridge_test, 4),
        'MAE Test':         round(mae_ridge_test, 4),
        'R² Test':          round(r2_ridge_test, 4),
        'Mejores hiperparámetros': str(grid_ridge.best_params_),
    },
    {
        'Modelo':           'Lasso',
        'RMSE CV (media)':  rmse_cv_lasso,
        'RMSE CV (std)':    std_cv_lasso,
        'RMSE Test':        round(rmse_lasso_test, 4),
        'MAE Test':         round(mae_lasso_test, 4),
        'R² Test':          round(r2_lasso_test, 4),
        'Mejores hiperparámetros': str(grid_lasso.best_params_),
    },
    {
        'Modelo':           'Polinomial + Ridge',
        'RMSE CV (media)':  rmse_cv_poly_ridge,
        'RMSE CV (std)':    std_cv_poly_ridge,
        'RMSE Test':        round(rmse_poly_ridge_test, 4),
        'MAE Test':         round(mae_poly_ridge_test, 4),
        'R² Test':          round(r2_poly_ridge_test, 4),
        'Mejores hiperparámetros': str(grid_poly_ridge.best_params_),
    },
])
resultados.set_index('Modelo', inplace=True)
resultados

In [ ]:
# Identificar el mejor modelo según RMSE CV promedio
mejor_nombre = resultados['RMSE CV (media)'].idxmin()
print(f'Mejor modelo según RMSE CV promedio: {mejor_nombre}')
print()
print(resultados.loc[mejor_nombre])

**Argumentación de la selección:**

El modelo seleccionado es aquel que combina el menor RMSE en validación cruzada con la menor desviación estándar, garantizando tanto precisión como estabilidad. En el contexto de AlpesHearth, donde el score de riesgo cardiovascular puede influir en decisiones clínicas, priorizar la estabilidad es fundamental: un modelo que funciona de forma consistente en distintos grupos de pacientes es más confiable que uno que tiene un desempeño excepcional en algunos casos pero falla en otros.

Si hay empate en RMSE, se prefiere el modelo más simple (menor número de parámetros) por su mayor interpretabilidad y facilidad de mantenimiento en producción.

---

## Actividad 6: Intervalos de confianza mediante Bootstrapping

Utilizamos el mejor modelo seleccionado para estimar intervalos de confianza al 95% sobre las métricas de desempeño en el conjunto de test. El procedimiento consiste en generar 500 remuestreos con reemplazo del conjunto de test, calcular RMSE, MAE y R² en cada uno, y construir los percentiles del 2.5% y 97.5%.

A diferencia de la validación cruzada (que evalúa sobre particiones del entrenamiento), el bootstrapping sobre el test permite estimar la variabilidad del desempeño **sobre datos no vistos**, lo que refleja mejor la incertidumbre real del modelo en producción.

In [ ]:
# Seleccionar el mejor modelo (ajustar la variable según el resultado de la Actividad 5)
# Opciones: best_poly, best_ridge, best_lasso, best_poly_ridge
mejor_modelo_final = best_ridge   # <-- reemplazar con el modelo ganador de la tabla comparativa

n_iterations = 500
bootstrap_rmse = []
bootstrap_mae  = []
bootstrap_r2   = []

np.random.seed(42)
for _ in range(n_iterations):
    X_resample, y_resample = resample(X_test, y_test, replace=True, n_samples=len(X_test))
    pred = mejor_modelo_final.predict(X_resample)
    bootstrap_rmse.append(np.sqrt(mean_squared_error(y_resample, pred)))
    bootstrap_mae.append(mean_absolute_error(y_resample, pred))
    bootstrap_r2.append(r2_score(y_resample, pred))

In [ ]:
alpha_ci = 0.95
p_lower  = ((1.0 - alpha_ci) / 2.0) * 100   # 2.5
p_upper  = (alpha_ci + (1.0 - alpha_ci) / 2.0) * 100  # 97.5

print('Intervalos de confianza al 95% — Bootstrapping (500 iteraciones)')
print(f'{"Métrica":<8} {"Media":>10} {"Std":>10} {"IC 2.5%":>12} {"IC 97.5%":>12}')
print('-' * 55)
for nombre, stats in [('RMSE', bootstrap_rmse), ('MAE', bootstrap_mae), ('R²', bootstrap_r2)]:
    lower = np.percentile(stats, p_lower)
    upper = np.percentile(stats, p_upper)
    media = np.mean(stats)
    std   = np.std(stats)
    print(f'{nombre:<8} {media:>10.4f} {std:>10.4f} {lower:>12.4f} {upper:>12.4f}')

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, datos, titulo in zip(axes,
                              [bootstrap_rmse, bootstrap_mae, bootstrap_r2],
                              ['RMSE', 'MAE', 'R²']):
    sns.histplot(datos, kde=True, ax=ax, color='steelblue')
    ax.axvline(np.percentile(datos, p_lower), color='red',   linestyle='--', label='IC 2.5%')
    ax.axvline(np.percentile(datos, p_upper), color='red',   linestyle='--', label='IC 97.5%')
    ax.axvline(np.mean(datos),               color='black', linestyle='-',  label='Media')
    ax.set_title(f'Distribución Bootstrap — {titulo}')
    ax.set_xlabel(titulo)
    ax.set_ylabel('Frecuencia')
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

**Interpretación de los intervalos de confianza:**

Un intervalo estrecho indica que el desempeño del modelo es estable y poco sensible a variaciones en el conjunto de test; es decir, el modelo generaliza de forma consistente sobre distintas muestras de pacientes. Un intervalo amplio sugiere mayor incertidumbre: el modelo puede funcionar significativamente mejor o peor dependiendo de qué pacientes sean evaluados.

En el contexto clínico de AlpesHearth, es deseable que el intervalo de confianza del RMSE sea lo más estrecho posible, ya que esto garantiza que las predicciones del score de riesgo cardiovascular son confiables y reproducibles en distintos grupos de pacientes, independientemente de su composición específica.

---

## Análisis de resultados

### Análisis cuantitativo

**¿Cuál modelo obtuvo el mejor desempeño en el conjunto de test?**

> *Completar con los valores de la tabla comparativa. Identificar el modelo con menor RMSE en test y mayor R².*

---

**¿Coincide el mejor desempeño en test con el mejor promedio en validación cruzada?**

> Si no coincide, la explicación más probable es la variabilidad inherente a la partición aleatoria de train/test. La validación cruzada promedia múltiples particiones y suaviza las fluctuaciones, mientras que el RMSE en test depende de un único subconjunto de 20% de los datos. Además, el proceso de GridSearchCV puede incurrir en un leve *overfitting del pipeline de selección* si el espacio de búsqueda es muy grande.

---

**¿El modelo con mejor métrica promedio es necesariamente el más adecuado?**

> No. Un modelo con RMSE CV promedio ligeramente menor pero con desviación estándar alta indica inestabilidad: en algunos folds rinde bien y en otros no. Un modelo con RMSE CV marginalmente mayor pero con std pequeño es más predecible y confiable para despliegue en producción. En el sector salud, donde se toman decisiones sobre pacientes reales, la estabilidad es preferible a ganancias marginales en precisión.

---

**¿Cómo cambia el error con la complejidad según las curvas de validación?**

> Las curvas muestran el trade-off sesgo-varianza clásico: al aumentar el grado del polinomio, el error en entrenamiento disminuye progresivamente (el modelo se ajusta cada vez mejor). Sin embargo, a partir de cierto grado (típicamente 3 o 4 con este tamaño de dataset), el error en validación cruzada comienza a aumentar o su desviación estándar se dispara. Este punto de inflexión delimita el grado óptimo y evidencia el inicio del sobreajuste: el modelo captura ruido específico del entrenamiento y pierde capacidad de generalización.

---

**¿Cómo afecta la regularización la magnitud y estabilidad de los coeficientes?**

> Ridge reduce la magnitud de todos los coeficientes de forma proporcional sin anular ninguno, produciendo un modelo más estable frente a la colinealidad entre variables clínicas correlacionadas (por ejemplo, BMI y circunferencia abdominal, o colesterol total y LDL estimado). Lasso, al anular algunos coeficientes, produce un modelo más parsimonioso con menor número de parámetros activos, lo que reduce la varianza del modelo.

---

**¿Los intervalos de confianza bootstrap sugieren estabilidad o alta variabilidad?**

> *Completar con los valores numéricos obtenidos. Un intervalo cuya amplitud sea inferior al 15-20% del valor central del RMSE se considera razonablemente estable para un modelo de scoring clínico.*

---

### Análisis cualitativo

**¿Qué variables fueron seleccionadas como más relevantes por Lasso?**

> *Completar con los resultados de la revisión de coeficientes. Las variables con mayor coeficiente absoluto son los factores de riesgo que el modelo considera más determinantes para el score cardiovascular. Candidatos esperados: Age, Systolic BP, Fasting Blood Sugar, Total Cholesterol, Smoking Status.*

---

**¿Qué interpretación práctica tienen los coeficientes del modelo final?**

> Cada coeficiente representa el cambio esperado en el `CVD Risk Score` por cada unidad de variación en la variable correspondiente (tras el escalamiento), manteniendo el resto constante. Coeficientes positivos indican factores que incrementan el riesgo (e.g., presión sistólica elevada, glucosa en ayuno alta). Coeficientes negativos indican efectos protectores (e.g., HDL alto reduce el riesgo cardiovascular). Esto permite a los clínicos de AlpesHearth priorizar intervenciones sobre los factores de mayor peso.

---

**¿Existen diferencias relevantes entre el modelo más preciso y el más interpretable?**

> El modelo polinomial regularizado puede tener menor RMSE, pero genera decenas o cientos de términos polinomiales cuya interpretación clínica directa es compleja. Lasso sobre variables originales produce coeficientes que corresponden directamente a variables clínicas medibles, facilitando la comunicación con equipos médicos. En AlpesHearth, si la ganancia de precisión del modelo complejo es marginal, el modelo más interpretable puede tener mayor valor práctico.

---

**¿Qué decisiones estratégicas podría tomar AlpesHearth?**

> - Priorizar intervenciones preventivas en pacientes con score predicho alto.
> - Las variables con mayor peso en Lasso identifican factores de riesgo modificables, orientando programas de reducción de riesgo focalizados (control de presión arterial, manejo de glucosa, etc.).
> - Las variables con coeficiente 0 en Lasso podrían considerarse prescindibles en la rutina de toma de datos, reduciendo costos de medición.
> - La estabilidad del modelo (validada con bootstrapping) respalda su uso confiable en distintas cohortes de pacientes.

---

**¿Mayor precisión implica necesariamente mayor valor para la organización?**

> No. Un modelo opaco o inestable puede generar desconfianza entre los clínicos y bajo nivel de adopción. En el contexto clínico, un modelo moderadamente preciso, estable e interpretable genera más valor que uno ligeramente más preciso pero incomprensible o de difícil mantenimiento. La confianza del equipo médico en el modelo es un factor clave para su adopción real.

---

**¿Un modelo más complejo genera mayor valor empresarial?**

> No necesariamente. La complejidad tiene costos directos: mayor dificultad de mantenimiento, necesidad de más datos para recalibración, mayor riesgo de sobreajuste en producción, menor facilidad de auditoría clínica y regulatoria. Si la mejora en RMSE es marginal respecto al modelo simple, el costo adicional de complejidad no se justifica.

---

### Reflexión conceptual

**¿Qué relación observas entre complejidad, generalización y estabilidad?**

> A mayor complejidad del modelo (más grados polinomiales, más parámetros), la capacidad de ajustar los datos de entrenamiento aumenta, pero la generalización puede degradarse (sobreajuste) y la estabilidad disminuye (alta varianza entre folds). La regularización actúa como mecanismo que controla esta complejidad, permitiendo modelos más ricos sin sacrificar la capacidad de generalización. El balance óptimo se encuentra con la búsqueda sistemática de hiperparámetros y la validación cruzada.

---

**¿Qué fuentes de sesgo podrían estar presentes?**

> - **Sesgo de selección:** si la muestra de pacientes no es representativa de la población objetivo de AlpesHearth (e.g., sobrerepresentación de ciertos grupos etarios, sexos o niveles de actividad física).
> - **Sesgo de medición:** errores sistemáticos en la recolección de variables clínicas (inconsistencias en fechas, formatos de presión arterial, unidades) que ya se identificaron en el Laboratorio 1.
> - **Sesgo histórico:** si los patrones de atención médica del periodo de los datos no son representativos de la práctica clínica actual.
> - **Sesgo de variable omitida:** factores de riesgo relevantes (e.g., estrés, dieta) no capturados en el dataset.

---

**Si el tamaño de muestra fuera mayor, ¿esperarías cambios en la estabilidad?**

> Sí. Con más datos, la variabilidad entre particiones en la validación cruzada disminuye, los intervalos de confianza del bootstrapping se estrechan y el modelo es menos sensible a valores atípicos. Esto permitiría explorar grados polinomiales más altos o espacios de búsqueda de hiperparámetros más amplios con menor riesgo de sobreajuste. Adicionalmente, con mayor muestra el modelo Lasso podría seleccionar un conjunto de variables más estable entre distintas particiones.

---

## Uso de herramientas de IA generativa

**Herramienta utilizada:** Claude (Anthropic) — asistencia en generación inicial de estructura del notebook y apoyo conceptual.

**Tipo de uso:** Generación de esqueleto inicial del código, ayuda conceptual sobre `validation_curve` y bootstrapping, sugerencias de estructura para las secciones de análisis.

**Prompts utilizados:**

- *'Genera el esqueleto de un notebook para regresión polinomial regularizada siguiendo el estilo de notebooks educativos con Pipeline y GridSearchCV en sklearn, adaptado al dataset cardiovascular de AlpesHearth con columnas específicas como Age, Systolic BP, HDL, etc.'*
- *'¿Cómo se usa `validation_curve` dentro de un Pipeline de sklearn para analizar el efecto del grado polinomial sobre el RMSE cuando el parámetro está anidado dentro de un ColumnTransformer?'*
- *'Explica el procedimiento de bootstrapping sobre el conjunto de test para estimar intervalos de confianza al 95% para RMSE, MAE y R², con 500 iteraciones.'*

**Análisis crítico del resultado:**

- *¿Qué partes fueron correctas y útiles?* La estructura general del notebook fue coherente con el estilo de los notebooks de referencia del curso. Las explicaciones conceptuales sobre el trade-off sesgo-varianza y la selección de variables por Lasso fueron precisas y útiles como base.

- *¿Qué errores o limitaciones se identificaron?* El código inicial asumía variables genéricas; fue necesario adaptarlo completamente a las columnas reales del dataset (`CVD Risk Score`, `Systolic BP`, `HDL`, etc.). El espacio de búsqueda de `alpha` fue ajustado tras observar el rango del score objetivo. Además, la gestión de columnas a descartar (`CVD Risk Level`, `Blood Pressure (mmHg)`) debió ser definida manualmente.

**Aportes propios:**

- Análisis de los datos reales para determinar qué columnas usar como predictores y cuáles descartar.
- Decisión de eliminar `CVD Risk Level` para evitar fuga de datos (data leakage).
- Ajuste de los rangos de hiperparámetros según el rango y distribución real del `CVD Risk Score`.
- Redacción de todo el análisis de resultados con interpretación clínica específica al contexto de AlpesHearth.
- Verificación y adaptación del código de bootstrapping para incluir MAE y R² con formato de tabla.